In [ ]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 7.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 22.3 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=c4f5cba91a11989b98aed88725eac1b59960d0b070e182b7820b730e1705ee95
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679802 sha256=ece846fd8b8132e248a0f5fac03ab2da309c25ee8176701a0c79c79484a4d079
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [ ]:
import numpy as np
import pandas as pd
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

In [ ]:
def run_benchmark(n_data, dim, metric_name, k=10):
    np.random.seed(42)
    X = np.random.random((n_data, dim)).astype(np.float32)
    query = np.random.random((1, dim)).astype(np.float32)

    if metric_name == 'angular':
        X_use = X.copy()
        q_use = query.copy()
        faiss.normalize_L2(X_use)
        faiss.normalize_L2(q_use)
    else:
        X_use = X
        q_use = query

    # 0. Ground Truth untuk hitung Recall
    if metric_name == 'euclidean':
        exact_index = faiss.IndexFlatL2(dim)
    else:
        exact_index = faiss.IndexFlatIP(dim)
    exact_index.add(X_use)
    _, exact_idx = exact_index.search(q_use, k)
    exact_set = set(exact_idx[0])

    # 1. ANNOY
    ann_metric = 'euclidean' if metric_name == 'euclidean' else 'angular'
    ann_index = AnnoyIndex(dim, ann_metric)

    start = time.time()
    for i in range(n_data):
        ann_index.add_item(i, X_use[i])
    ann_index.build(10)
    ann_build = time.time() - start

    start = time.time()
    ann_idx = ann_index.get_nns_by_vector(q_use[0], k)
    ann_query = time.time() - start
    ann_recall = len(set(ann_idx) & exact_set) / k

    # 2. FAISS IVF+PQ (Pattern Lab 2)
    nlist = 10 if n_data <= 1000 else 100
    m = 1 if dim == 2 else 5

    if metric_name == 'euclidean':
        quantizer = faiss.IndexFlatL2(dim)
        faiss_index = faiss.IndexIVFPQ(quantizer, dim, nlist, m, 8, faiss.METRIC_L2)
    else:
        quantizer = faiss.IndexFlatIP(dim)
        faiss_index = faiss.IndexIVFPQ(quantizer, dim, nlist, m, 8, faiss.METRIC_INNER_PRODUCT)

    start = time.time()
    faiss_index.train(X_use)
    faiss_index.add(X_use)
    faiss_build = time.time() - start

    faiss_index.nprobe = 5
    start = time.time()
    _, faiss_idx = faiss_index.search(q_use, k)
    faiss_query = time.time() - start
    faiss_recall = len(set(faiss_idx[0]) & exact_set) / k

    # 3. HNSW
    hnsw_space = 'l2' if metric_name == 'euclidean' else 'cosine'
    hnsw_index = hnswlib.Index(space=hnsw_space, dim=dim)

    start = time.time()
    hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
    hnsw_index.add_items(X_use)
    hnsw_build = time.time() - start

    hnsw_index.set_ef(50)
    start = time.time()
    hnsw_idx, _ = hnsw_index.knn_query(q_use, k=k)
    hnsw_query = time.time() - start
    hnsw_recall = len(set(hnsw_idx[0]) & exact_set) / k

    return {
        "Jumlah Data/Dimensi": f"{n_data}/{dim}D ({metric_name})",
        "ANNOY": f"Build: {ann_build:.4f}s | Query: {ann_query*1000:.4f}ms | Recall: {ann_recall:.2f}",
        "FAISS": f"Build: {faiss_build:.4f}s | Query: {faiss_query*1000:.4f}ms | Recall: {faiss_recall:.2f}",
        "HNSW": f"Build: {hnsw_build:.4f}s | Query: {hnsw_query*1000:.4f}ms | Recall: {hnsw_recall:.2f}"
    }

In [ ]:
scenarios = [
    (1000, 2, 'euclidean'),
    (1000, 2, 'angular'),
    (1_000_000, 5, 'euclidean'),
    (1_000_000, 5, 'angular'),
]

results = []
for n_data, dim, metric in scenarios:
    print(f"Menjalankan skenario {n_data} data, {dim}D, metric={metric}...")
    res = run_benchmark(n_data, dim, metric, k=10)
    results.append(res)

df_hasil = pd.DataFrame(results)
display(df_hasil)

Menjalankan skenario 1000 data, 2D, metric=euclidean...
Menjalankan skenario 1000 data, 2D, metric=angular...
Menjalankan skenario 1000000 data, 5D, metric=euclidean...
Menjalankan skenario 1000000 data, 5D, metric=angular...


,Jumlah Data/Dimensi,ANNOY,FAISS,HNSW
0,1000/2D (euclidean),Build: 0.0847s | Query: 0.0863ms | Recall: 0.90,Build: 0.1652s | Query: 0.0904ms | Recall: 0.90,Build: 0.1635s | Query: 0.0741ms | Recall: 1.00
1,1000/2D (angular),Build: 0.0987s | Query: 0.0706ms | Recall: 1.00,Build: 0.0404s | Query: 0.0796ms | Recall: 0.40,Build: 0.1205s | Query: 0.0665ms | Recall: 1.00
2,1000000/5D (euclidean),Build: 20.9055s | Query: 0.1223ms | Recall: 1.00,Build: 29.7507s | Query: 0.7484ms | Recall: 1.00,Build: 166.1223s | Query: 0.1216ms | Recall: 1.00
3,1000000/5D (angular),Build: 27.9365s | Query: 0.0925ms | Recall: 1.00,Build: 29.2234s | Query: 0.7741ms | Recall: 0.60,Build: 156.2414s | Query: 0.0916ms | Recall: 1.00
